# Distributed SNOWPACK Data Processing with Dask

This notebook demonstrates how to use Dask to process gridded SNOWPACK `.pro` files stored in cloud storage (S3, GCS, Azure, HTTP, etc.) across multiple machines, filter for north-facing slopes, and create geospatial visualizations.

## 📚 Table of Contents

1. [Setup & Installation](#setup)
2. [Dask Cluster Configuration](#dask-cluster)
3. [Storage Backend Setup](#storage)
4. [Data Loading & Processing](#processing)
5. [Filtering & Analysis](#analysis)
6. [Visualization](#visualization)
7. [Complete Workflow Example](#complete-example)

---

## 1. Setup & Installation <a id='setup'></a>

First, let's install and import all required packages.

In [ ]:
# Install required packages (run once)
# Uncomment the lines below if packages are not installed

# !pip install xsnow dask distributed
# !pip install s3fs gcsfs adlfs  # Cloud storage backends
# !pip install geopandas matplotlib contextily  # Visualization
# !pip install siphon  # For THREDDS catalogs
# !pip install beautifulsoup4 requests  # For HTTP directory listings

In [ ]:
# Import libraries
import dask
from dask.distributed import Client, LocalCluster, progress
import xarray as xr
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import contextily as ctx
import s3fs
import warnings
from pathlib import Path
import os

from xsnow import read
from xsnow.selectors import TimeSelector, LocationSelector, SlopeSelector

# Configure display
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (12, 8)
%matplotlib inline

print("✅ All imports successful!")

---
## 2. Dask Cluster Configuration <a id='dask-cluster'></a>

Set up a Dask cluster. You can choose between:
- **Local cluster** (for testing on your laptop)
- **Remote cluster** (connect to existing cluster)
- **Cloud cluster** (AWS, GCP, Azure, Kubernetes)

### Option A: Local Cluster (Recommended for Getting Started)

In [ ]:
def setup_local_cluster(n_workers=4, threads_per_worker=2, memory_limit='4GB'):
    """
    Set up a local Dask cluster for testing.
    
    Parameters:
    -----------
    n_workers : int
        Number of worker processes
    threads_per_worker : int
        Threads per worker
    memory_limit : str
        Memory limit per worker (e.g., '4GB', '8GB')
    
    Returns:
    --------
    client : dask.distributed.Client
    cluster : dask.distributed.LocalCluster
    """
    cluster = LocalCluster(
        n_workers=n_workers,
        threads_per_worker=threads_per_worker,
        memory_limit=memory_limit,
        dashboard_address=':8787'
    )
    client = Client(cluster)
    
    print(f"🚀 Dask Dashboard: {client.dashboard_link}")
    print(f"📊 Workers: {n_workers}")
    print(f"💾 Memory per worker: {memory_limit}")
    
    return client, cluster

# Start local cluster
client, cluster = setup_local_cluster(
    n_workers=4,
    threads_per_worker=2,
    memory_limit='4GB'
)

# Display cluster info
client

### Option B: Remote/Cloud Cluster

Connect to an existing Dask scheduler running on remote infrastructure.

In [ ]:
# Uncomment and configure for your infrastructure

# # Connect to existing scheduler
# client = Client('tcp://scheduler-address:8786')

# # Or for Kubernetes
# from dask_kubernetes import KubeCluster
# cluster = KubeCluster.from_yaml('worker-spec.yaml')
# cluster.scale(20)
# client = Client(cluster)

# # Or for SLURM
# from dask_jobqueue import SLURMCluster
# cluster = SLURMCluster(
#     cores=24,
#     memory="96GB",
#     walltime="04:00:00",
#     queue="regular"
# )
# cluster.scale(jobs=10)
# client = Client(cluster)

---
## 3. Storage Backend Setup <a id='storage'></a>

Configure access to your data storage. This notebook supports multiple backends.

### Configure Your Storage Backend

In [ ]:
# ============================================================================
# CONFIGURATION: Choose your storage backend
# ============================================================================

STORAGE_TYPE = 'local'  # Options: 'local', 's3', 'gcs', 'azure', 'http'

# Configure based on your storage type
if STORAGE_TYPE == 'local':
    DATA_PATHS = ['/path/to/your/snowpack/*.pro']
    STORAGE_OPTIONS = {}

elif STORAGE_TYPE == 's3':
    BUCKET = 'my-snowpack-bucket'
    PREFIX = 'gridded_pros/2025/winter'
    DATA_PATHS = None  # Will be populated below
    STORAGE_OPTIONS = {
        # 'key': 'AWS_ACCESS_KEY_ID',
        # 'secret': 'AWS_SECRET_ACCESS_KEY',
        # Or use environment variables / IAM roles
    }

elif STORAGE_TYPE == 'gcs':
    BUCKET = 'my-snowpack-bucket'
    PREFIX = 'gridded_pros/2025/winter'
    DATA_PATHS = None
    STORAGE_OPTIONS = {
        # 'token': 'path/to/credentials.json'
    }

elif STORAGE_TYPE == 'azure':
    CONTAINER = 'my-snowpack-container'
    PREFIX = 'gridded_pros/2025/winter'
    DATA_PATHS = None
    STORAGE_OPTIONS = {
        # 'account_name': 'mystorageaccount',
        # 'account_key': 'YOUR_ACCOUNT_KEY'
    }

elif STORAGE_TYPE == 'http':
    DATA_PATHS = [
        'https://data.example.com/snowpack/station_001.pro',
        'https://data.example.com/snowpack/station_002.pro',
    ]
    STORAGE_OPTIONS = {
        # 'headers': {'Authorization': 'Bearer YOUR_TOKEN'}
    }

print(f"✅ Storage configured: {STORAGE_TYPE}")

### S3: List Files in Bucket

In [ ]:
if STORAGE_TYPE == 's3':
    def list_s3_files(bucket, prefix='', max_files=None):
        """List .pro files in S3 bucket."""
        fs = s3fs.S3FileSystem(**STORAGE_OPTIONS)
        
        path = f"{bucket}/{prefix}" if prefix else bucket
        all_files = fs.glob(f"{path}/**/*.pro")
        
        # Convert to s3:// URIs
        s3_paths = [f"s3://{f}" for f in all_files]
        
        if max_files:
            s3_paths = s3_paths[:max_files]
        
        print(f"📁 Found {len(s3_paths)} files in s3://{path}")
        return s3_paths
    
    DATA_PATHS = list_s3_files(BUCKET, PREFIX, max_files=100)
    print(f"\n📝 First 5 files:")
    for path in DATA_PATHS[:5]:
        print(f"   {path}")

### HTTP: Discover Files from Directory Listing

In [ ]:
if STORAGE_TYPE == 'http':
    import requests
    from bs4 import BeautifulSoup
    from urllib.parse import urljoin
    
    def list_http_directory(base_url, extension='.pro'):
        """Parse HTTP directory listing to find files."""
        headers = STORAGE_OPTIONS.get('headers', {})
        response = requests.get(base_url, headers=headers)
        response.raise_for_status()
        
        soup = BeautifulSoup(response.text, 'html.parser')
        links = soup.find_all('a')
        
        files = []
        for link in links:
            href = link.get('href')
            if href and href.endswith(extension):
                full_url = urljoin(base_url, href)
                files.append(full_url)
        
        print(f"📁 Found {len(files)} files at {base_url}")
        return files
    
    # Uncomment to discover files from directory listing
    # BASE_URL = 'https://data.example.com/snowpack/2024/'
    # DATA_PATHS = list_http_directory(BASE_URL)

---
## 4. Data Loading & Processing <a id='processing'></a>

Load SNOWPACK data using xsnow with lazy loading for efficient memory usage.

In [ ]:
def load_snowpack_data(paths, lazy=True, client=None):
    """
    Load SNOWPACK data with Dask.
    
    Parameters:
    -----------
    paths : list
        List of file paths/URIs
    lazy : bool
        Use lazy loading (recommended)
    client : dask.distributed.Client
        Dask client for distributed processing
    
    Returns:
    --------
    ds : xsnowDataset
    """
    print(f"📂 Loading {len(paths)} files...")
    print(f"   Lazy loading: {lazy}")
    
    ds = read(
        paths,
        lazy=lazy,
        storage_options=STORAGE_OPTIONS if STORAGE_OPTIONS else None
    )
    
    if ds is None:
        raise ValueError("Failed to load data")
    
    print(f"\n✅ Dataset loaded successfully!")
    print(f"   Dimensions: {dict(ds.sizes)}")
    print(f"   Variables: {list(ds.data.data_vars)}")
    
    return ds

# Load data
ds = load_snowpack_data(DATA_PATHS, lazy=True, client=client)

### Inspect Dataset

In [ ]:
# Display dataset structure
ds.data

In [ ]:
# Check coordinates
print("📍 Coordinates:")
print(f"   Locations: {ds.sizes['location']}")
print(f"   Time steps: {ds.sizes['time']}")
print(f"   Slopes: {ds.sizes.get('slope', 1)}")
print(f"   Max layers: {ds.sizes['layer']}")

# Check spatial extent
if 'latitude' in ds.data.coords and 'longitude' in ds.data.coords:
    print(f"\n🌍 Spatial Extent:")
    print(f"   Latitude: {float(ds.data.latitude.min()):.3f} to {float(ds.data.latitude.max()):.3f}")
    print(f"   Longitude: {float(ds.data.longitude.min()):.3f} to {float(ds.data.longitude.max()):.3f}")

---
## 5. Filtering & Analysis <a id='analysis'></a>

Filter for north-facing slopes and compute snow height statistics.

### Filter for North-Facing Slopes

In [ ]:
def filter_by_aspect(ds, aspect_range=(315, 45), aspect_name='North'):
    """
    Filter dataset by slope aspect (azimuth).
    
    Parameters:
    -----------
    ds : xsnowDataset
        Input dataset
    aspect_range : tuple
        (min_azimuth, max_azimuth) in degrees
        North: (315, 45), East: (45, 135), South: (135, 225), West: (225, 315)
    aspect_name : str
        Name for display
    
    Returns:
    --------
    ds_filtered : xsnowDataset
    """
    if 'slope_azimuth' not in ds.data.coords:
        print("⚠️ No slope_azimuth coordinate found - skipping aspect filtering")
        return ds
    
    min_az, max_az = aspect_range
    
    print(f"🧭 Filtering for {aspect_name}-facing slopes ({min_az}° - {max_az}°)...")
    
    # Handle wrapping around 0/360
    if min_az > max_az:
        mask = (ds.data.slope_azimuth >= min_az) | (ds.data.slope_azimuth <= max_az)
    else:
        mask = (ds.data.slope_azimuth >= min_az) & (ds.data.slope_azimuth <= max_az)
    
    ds_filtered = ds.data.where(mask, drop=True)
    
    print(f"   Before: {ds.sizes.get('slope', 'N/A')} slopes")
    print(f"   After: {ds_filtered.sizes.get('slope', 'N/A')} slopes")
    
    return ds.__class__(ds_filtered)

# Filter for north-facing slopes
ds_north = filter_by_aspect(ds, aspect_range=(315, 45), aspect_name='North')

### Compute Snow Height

In [ ]:
# Compute HS (snow height) if not already present
if 'HS' not in ds_north.data:
    print("📏 Computing snow height...")
    ds_north = ds_north.compute_hs()
    print("   ✅ HS computed")
else:
    print("✅ HS already present in dataset")

# Check HS variable
ds_north.data['HS']

### Extract Latest Snow Height Data

In [ ]:
def extract_snow_height_data(ds):
    """
    Extract snow height data as pandas DataFrame.
    
    Parameters:
    -----------
    ds : xsnowDataset
        Dataset with HS variable
    
    Returns:
    --------
    df : pandas.DataFrame
        DataFrame with location, coordinates, and HS
    """
    print("📊 Extracting snow height data...")
    
    # Get latest timestamp
    latest_time = ds.data.time.values[-1]
    print(f"   Using data from: {pd.Timestamp(latest_time)}")
    
    # Select latest time
    hs_latest = ds.data['HS'].sel(time=latest_time)
    
    # Average across slope dimension if present
    if 'slope' in hs_latest.dims:
        hs_mean = hs_latest.mean(dim='slope')
    else:
        hs_mean = hs_latest
    
    # Compute (triggers Dask computation)
    print("   ⏳ Computing... (this may take a while)")
    with progress:
        hs_values = hs_mean.compute()
    
    # Extract coordinates
    lats = ds.data.latitude.values
    lons = ds.data.longitude.values
    alts = ds.data.altitude.values if 'altitude' in ds.data.coords else np.full(len(lats), np.nan)
    
    # Get location names
    location_mapping = ds.data.location.attrs.get('mapping', {})
    location_names = [location_mapping.get(i, f"Location_{i}") 
                     for i in range(len(hs_values))]
    
    # Create DataFrame
    df = pd.DataFrame({
        'location': location_names,
        'latitude': lats,
        'longitude': lons,
        'altitude': alts,
        'snow_height_m': hs_values.values,
        'snow_height_cm': hs_values.values * 100,
        'timestamp': pd.Timestamp(latest_time)
    })
    
    # Remove NaN values
    df = df.dropna(subset=['snow_height_cm'])
    
    print(f"\n✅ Extracted data for {len(df)} locations")
    print(f"   Mean HS: {df['snow_height_cm'].mean():.1f} cm")
    print(f"   Max HS: {df['snow_height_cm'].max():.1f} cm")
    print(f"   Min HS: {df['snow_height_cm'].min():.1f} cm")
    
    return df

# Extract data
df_snow = extract_snow_height_data(ds_north)

### Preview Results

In [ ]:
# Display first few rows
df_snow.head(10)

In [ ]:
# Statistical summary
df_snow[['snow_height_cm', 'altitude']].describe()

In [ ]:
# Top 10 locations with most snow
print("🏔️ Top 10 locations with most snow:\n")
df_snow.nlargest(10, 'snow_height_cm')[['location', 'snow_height_cm', 'altitude', 'latitude', 'longitude']]

---
## 6. Visualization <a id='visualization'></a>

Create geospatial visualizations of snow height distribution.

### Create Interactive Map

In [ ]:
def create_snow_map(df, figsize=(15, 12), cmap='Blues', add_basemap=True):
    """
    Create a map visualization of snow height.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        DataFrame with location, lat, lon, snow_height_cm
    figsize : tuple
        Figure size
    cmap : str
        Colormap name
    add_basemap : bool
        Add satellite basemap
    
    Returns:
    --------
    fig : matplotlib.figure.Figure
    """
    print("🗺️ Creating map...")
    
    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitude, df.latitude),
        crs='EPSG:4326'
    )
    
    # Reproject to Web Mercator for basemap
    if add_basemap:
        gdf = gdf.to_crs('EPSG:3857')
    
    # Create figure
    fig, ax = plt.subplots(figsize=figsize)
    
    # Plot snow height
    scatter = gdf.plot(
        column='snow_height_cm',
        ax=ax,
        cmap=cmap,
        legend=True,
        markersize=100,
        edgecolor='black',
        linewidth=0.5,
        alpha=0.8,
        legend_kwds={
            'label': 'Snow Height (cm)',
            'orientation': 'horizontal',
            'shrink': 0.8,
            'pad': 0.05
        }
    )
    
    # Add basemap
    if add_basemap:
        try:
            ctx.add_basemap(
                ax,
                source=ctx.providers.Esri.WorldImagery,
                alpha=0.6,
                attribution=False
            )
        except Exception as e:
            print(f"   ⚠️ Could not add basemap: {e}")
    
    # Styling
    timestamp = df['timestamp'].iloc[0]
    ax.set_title(
        f'Snow Height on North-Facing Slopes\n{timestamp.strftime("%Y-%m-%d %H:%M")}',
        fontsize=16,
        fontweight='bold',
        pad=20
    )
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--')
    
    # Add statistics
    stats_text = (
        f"Locations: {len(df)}\n"
        f"Mean: {df['snow_height_cm'].mean():.1f} cm\n"
        f"Max: {df['snow_height_cm'].max():.1f} cm\n"
        f"Min: {df['snow_height_cm'].min():.1f} cm"
    )
    ax.text(
        0.98, 0.98, stats_text,
        transform=ax.transAxes,
        fontsize=10,
        verticalalignment='top',
        horizontalalignment='right',
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.8)
    )
    
    plt.tight_layout()
    print("   ✅ Map created")
    
    return fig

# Create map
fig_map = create_snow_map(df_snow, add_basemap=True)
plt.show()

### Snow Height Distribution

In [ ]:
# Histogram of snow heights
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
ax1.hist(df_snow['snow_height_cm'], bins=30, edgecolor='black', alpha=0.7)
ax1.axvline(df_snow['snow_height_cm'].mean(), color='red', linestyle='--', linewidth=2, label='Mean')
ax1.axvline(df_snow['snow_height_cm'].median(), color='orange', linestyle='--', linewidth=2, label='Median')
ax1.set_xlabel('Snow Height (cm)', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Distribution of Snow Heights', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Box plot
ax2.boxplot(df_snow['snow_height_cm'], vert=True)
ax2.set_ylabel('Snow Height (cm)', fontsize=12)
ax2.set_title('Snow Height Box Plot', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

### Snow Height vs Elevation

In [ ]:
if 'altitude' in df_snow.columns and not df_snow['altitude'].isna().all():
    fig, ax = plt.subplots(figsize=(12, 6))
    
    scatter = ax.scatter(
        df_snow['altitude'],
        df_snow['snow_height_cm'],
        c=df_snow['snow_height_cm'],
        cmap='Blues',
        s=100,
        alpha=0.6,
        edgecolors='black',
        linewidth=0.5
    )
    
    ax.set_xlabel('Elevation (m)', fontsize=12)
    ax.set_ylabel('Snow Height (cm)', fontsize=12)
    ax.set_title('Snow Height vs Elevation', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    plt.colorbar(scatter, ax=ax, label='Snow Height (cm)')
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Altitude data not available")

---
## 7. Complete Workflow Example <a id='complete-example'></a>

Put it all together in a single workflow function.

In [ ]:
def complete_workflow(
    data_paths,
    aspect_range=(315, 45),
    output_dir='./output',
    dask_client=None
):
    """
    Complete workflow: load, process, analyze, visualize.
    
    Parameters:
    -----------
    data_paths : list
        List of file paths/URIs
    aspect_range : tuple
        Aspect filter (min, max) azimuth
    output_dir : str
        Directory to save outputs
    dask_client : dask.distributed.Client
        Dask client
    
    Returns:
    --------
    df : pandas.DataFrame
        Results dataframe
    """
    # Create output directory
    output_path = Path(output_dir)
    output_path.mkdir(exist_ok=True, parents=True)
    
    print("="*80)
    print("DISTRIBUTED SNOWPACK DATA PROCESSING WORKFLOW")
    print("="*80)
    
    # Step 1: Load data
    print("\n[1/5] Loading data...")
    ds = load_snowpack_data(data_paths, lazy=True, client=dask_client)
    
    # Step 2: Filter by aspect
    print("\n[2/5] Filtering by aspect...")
    ds_filtered = filter_by_aspect(ds, aspect_range=aspect_range)
    
    # Step 3: Compute snow height
    print("\n[3/5] Computing snow height...")
    if 'HS' not in ds_filtered.data:
        ds_filtered = ds_filtered.compute_hs()
    
    # Step 4: Extract statistics
    print("\n[4/5] Extracting statistics...")
    df = extract_snow_height_data(ds_filtered)
    
    # Save CSV
    csv_path = output_path / 'snow_height_results.csv'
    df.to_csv(csv_path, index=False)
    print(f"   💾 Saved to {csv_path}")
    
    # Step 5: Create visualizations
    print("\n[5/5] Creating visualizations...")
    fig_map = create_snow_map(df)
    map_path = output_path / 'snow_height_map.png'
    fig_map.savefig(map_path, dpi=300, bbox_inches='tight')
    print(f"   💾 Saved map to {map_path}")
    
    # Summary
    print("\n" + "="*80)
    print("SUMMARY")
    print("="*80)
    print(f"Total locations: {len(df)}")
    print(f"Mean snow height: {df['snow_height_cm'].mean():.1f} cm")
    print(f"Max snow height: {df['snow_height_cm'].max():.1f} cm")
    print(f"Min snow height: {df['snow_height_cm'].min():.1f} cm")
    print(f"Std dev: {df['snow_height_cm'].std():.1f} cm")
    
    print("\n🏔️ Top 5 locations with most snow:")
    print(df.nlargest(5, 'snow_height_cm')[['location', 'snow_height_cm', 'latitude', 'longitude']])
    
    print("\n✅ Workflow complete!")
    
    return df

# Run complete workflow
results_df = complete_workflow(
    data_paths=DATA_PATHS,
    aspect_range=(315, 45),  # North-facing
    output_dir='./snowpack_output',
    dask_client=client
)

---
## Cleanup

Close Dask cluster when done.

In [ ]:
# Close Dask cluster
client.close()
cluster.close()

print("✅ Dask cluster closed")

---
## Additional Examples

### Different Aspect Filters

In [ ]:
# Compare different aspects
aspects = {
    'North': (315, 45),
    'East': (45, 135),
    'South': (135, 225),
    'West': (225, 315)
}

aspect_results = {}

for name, range_val in aspects.items():
    print(f"\nProcessing {name}-facing slopes...")
    ds_aspect = filter_by_aspect(ds, aspect_range=range_val, aspect_name=name)
    
    if 'HS' not in ds_aspect.data:
        ds_aspect = ds_aspect.compute_hs()
    
    df_aspect = extract_snow_height_data(ds_aspect)
    aspect_results[name] = df_aspect['snow_height_cm'].mean()

# Plot comparison
fig, ax = plt.subplots(figsize=(10, 6))
aspects_list = list(aspect_results.keys())
values_list = list(aspect_results.values())

bars = ax.bar(aspects_list, values_list, color=['blue', 'green', 'red', 'orange'])
ax.set_ylabel('Mean Snow Height (cm)', fontsize=12)
ax.set_title('Mean Snow Height by Aspect', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}',
            ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

### Time Series Analysis (if multiple time steps)

In [ ]:
if ds_north.sizes['time'] > 1:
    print("📈 Analyzing time series...")
    
    # Compute mean HS over time
    hs_timeseries = ds_north.data['HS'].mean(dim=['location', 'slope'])
    
    with progress:
        hs_ts_computed = hs_timeseries.compute()
    
    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    
    times = pd.to_datetime(hs_ts_computed.time.values)
    values = hs_ts_computed.values * 100  # Convert to cm
    
    ax.plot(times, values, linewidth=2, marker='o', markersize=4)
    ax.set_xlabel('Date', fontsize=12)
    ax.set_ylabel('Mean Snow Height (cm)', fontsize=12)
    ax.set_title('Snow Height Time Series (North-Facing Slopes)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print("⚠️ Only one time step available - skipping time series analysis")

---
## Notes

- **Dashboard**: Monitor your Dask cluster at the dashboard link printed above
- **Memory**: Adjust `memory_limit` in cluster setup based on your data size
- **Workers**: Scale workers based on available compute resources
- **Storage**: This notebook supports S3, GCS, Azure, HTTP, and local storage
- **Authentication**: Set credentials via environment variables or IAM roles

## Next Steps

1. Customize aspect filters for your analysis
2. Add temporal analysis for seasonal trends
3. Export results to different formats (NetCDF, GeoTIFF)
4. Scale to cloud infrastructure for production workloads
5. Integrate with your existing data pipelines

---

**Questions?** Check the [xsnow documentation](https://xsnow.readthedocs.io) or [Dask documentation](https://docs.dask.org).